# Matched Product Captioning 
Since image quality can matter a lot in accurate identification of products, we wanted to explore captions quality on product marketing images. These images are likely to have no image quality distortions and fall in the training sets of VLMs. 

## Imports

In [ ]:
# data processing
import pandas as pd
import os
import sys
import json
import gc
from datetime import datetime

# progress bar
from tqdm.notebook import tqdm

# environment variables
from dotenv import load_dotenv

load_dotenv()

# setup
sys.path.append("../../")
pd.options.display.max_columns = None
pd.options.display.max_rows = None

# automatically reload modules
%load_ext autoreload
%autoreload 2

In [ ]:
# image proceessing
import requests
from PIL import Image
from io import BytesIO
import base64
import io

# VLMs
import torch
from openai import OpenAI
from google import genai
from google.genai import types

from transformers import MllamaForConditionalGeneration, AutoProcessor
from transformers import AutoModelForCausalLM

from scripts.molmo_captioner import generate_caption as get_molmo_caption
from scripts.llama_captioner import generate_caption as get_llama_caption

## Data Loading

In [ ]:
# load data
# check if we have any labeled data first
# captioned_file = (
#     "./final-output/combined-sample-25-images-captions_all-models_2025-08-19_16:08.json"
# )
captioned_file = None

combined_sample_dict = None
if captioned_file and os.path.isfile(captioned_file):
    with open(captioned_file, "r") as f:
        combined_sample_dict = json.load(f)
        LIMIT = len(combined_sample_dict)
else:
    # load matched products
    matched_product_df = pd.read_csv("matched-product-input.csv")
    matched_product_df["type"] = "matched-image"

    # load low- and high-quality images
    low_quality_df = pd.read_csv("./low-quality-images_08-19-25.csv")
    high_quality_df = pd.read_csv("./high-quality-images_08-19-25.csv")

    low_quality_df.fillna("", inplace=True)
    high_quality_df.fillna("", inplace=True)

    low_quality_df["type"] = "low-quality"
    high_quality_df["type"] = "high-quality"

    low_quality_df = low_quality_df.rename(columns={"vizwiz_url": "image_url"})
    high_quality_df = high_quality_df.rename(
        columns={"vizwiz_url": "image_url", "image_id": "id"}
    )

    # make object, product, brand, and variety columns lowercase and trimmed
    for col in ["object", "product", "brand", "variety"]:
        high_quality_df[col] = high_quality_df[col].str.lower().str.strip()
        low_quality_df[col] = low_quality_df[col].str.lower().str.strip()

    # pick specific images
    image_ids = [
        17784,
        5250,
        16851,
        16146,
        12643,
        18031,
        8125,
        4356,
        5395,
        12469,
        2004,
        5897,
        1236,
    ]
    combined_sample = pd.concat(
        [
            low_quality_df[
                [
                    "id",
                    "image_url",
                    "type",
                    "image_preview",
                    "object",
                    "product",
                    "brand",
                    "variety",
                ]
            ],
            high_quality_df[
                [
                    "id",
                    "image_url",
                    "type",
                    "image_preview",
                    "object",
                    "product",
                    "brand",
                    "variety",
                ]
            ],
            matched_product_df,
        ]
    )
    combined_sample.fillna("", inplace=True)
    combined_sample = combined_sample[combined_sample["id"].isin(image_ids)]
    combined_sample.reset_index(drop=True, inplace=True)
    combined_sample_dict = combined_sample.to_dict(orient="records")
    LIMIT = len(combined_sample_dict)

    # create a sample
    # LIMIT = 25

    # low_quality_sample = low_quality_df[
    #     (low_quality_df["unable_to_verify"] == "") & (low_quality_df["exclude?"] == "")
    # ].sample(n=LIMIT, random_state=42)
    # high_quality_sample = high_quality_df[(high_quality_df["exclude?"] == "")].sample(
    #     n=LIMIT, random_state=42
    # )
    # matched_product_sample = matched_product_df.sample(n=LIMIT, random_state=42)

    # combined_sample = pd.concat(
    #     [
    #         low_quality_sample[
    #             [
    #                 "id",
    #                 "image_url",
    #                 "type",
    #                 "image_preview",
    #                 "object",
    #                 "product",
    #                 "brand",
    #                 "variety",
    #             ]
    #         ],
    #         high_quality_sample[
    #             [
    #                 "id",
    #                 "image_url",
    #                 "type",
    #                 "image_preview",
    #                 "object",
    #                 "product",
    #                 "brand",
    #                 "variety",
    #             ]
    #         ],
    #         matched_product_sample,
    #     ]
    # )
    # combined_sample.fillna("", inplace=True)
    # combined_sample.reset_index(drop=True, inplace=True)

    # combined_sample_dict = combined_sample.to_dict(orient="records")

# show the data we're working with
print(f"Number of samples: {len(combined_sample_dict)}")
display(combined_sample.head())
combined_sample_dict[0]

## Prompt config
This notebook tests two prompts, currently:
1. The original VLM prompt we used for the ASSETS study
2. A revised prompt that encourages the model to detail product, brand, and details identified.

In [ ]:
from scripts.constants import get_prompt

VLM_ORIG_PROMPT = get_prompt()
# VLM_REV_PROMPT = (
#     "You are a helpful assistant who identifies products in images for blind and low-vision individuals. Follow these guidelines and only output the final description:\n"
#     "Step 1: Identify the product in the image.\n"
#     "Step 2: Identify crucial features about the product from Step 1, including:\n"
#     "-- Object type, such as can, bag, plastic container, etc.\n"
#     "-- Product type, such as prepared or frozen meal, seasoning mix, soda, coffee, etc.\n"
#     "-- Brand, such as Heinz, Coca-Cola, Starbucks, etc.\n"
#     "-- Variety, such as specific flavors, sizes, count of items, etc.\n"
#     "-- Visual features, such as color, shape, size, etc.\n"
#     "Step 3: Use clear, direct, and objective language. Do not use vague adjectives like 'large' or 'small', and vague adverbs like 'prominently' or 'clearly'.\n"
#     "Step 4: DO NOT mention camera artifacts (e.g., blur) or if an object is partially visible.\n"
#     "Step 5: DO NOT use introductory phrases (e.g., 'The image shows', 'The object is', 'The primary object is').\n\n"
#     "Output only the final description."
# )

# VLM_REV_PROMPT = (
#     "You are a helpful assistant who identifies products in images for blind and low-vision individuals. Follow these guidelines and only output the final description:\n"
#     "Step 1: Identify the product in the image.\n"
#     "Step 2: Identify crucial features about the product from Step 1, including:\n"
#     "-- Object type (can, bag, plastic container, etc.) \n"
#     "-- Product type (prepared or frozen meal, seasoning mix, soda, coffee) \n"
#     "-- Brand (Heinz, Coca-Cola, Starbucks, etc.) \n"
#     "-- Variety (specific flavors, sizes, count of items, etc.) \n"
#     "-- Visual features (color, shape, size, etc.) \n"
#     "Step 3: Use clear, direct, and objective language. Do not use vague adjectives like 'large' or 'small', or vague adverbs like 'prominently' or 'clearly'.\n"
#     "Step 4: DO NOT mention camera artifacts (e.g., blur) or if an object is partially visible.\n"
#     "Step 5: DO NOT use introductory phrases (e.g., 'The image shows', 'The object is', 'The primary object is').\n\n"
#     "Output only the final description."
# )

VLM_REV_PROMPT = (
    "You are a helpful assistant who identifies products in images for blind and low-vision individuals. Identify the product in the image while following these guidelines:\n"
    "1: Identify crucial features about the product, including:\n"
    "-- Object type (can, bag, plastic container, etc.) \n"
    "-- Product type (prepared or frozen meal, seasoning mix, soda, coffee) \n"
    "-- Brand (Heinz, Coca-Cola, Starbucks, etc.) \n"
    "-- Variety (specific flavors, sizes, count of items, etc.) \n"
    "-- Visual features (color, shape, size, etc.) \n"
    "2: Use clear, direct, and objective language. Do not use vague adjectives like 'large' or 'small', or vague adverbs like 'prominently' or 'clearly'.\n"
    "3: DO NOT mention camera artifacts (e.g., blur) or if an object is partially visible.\n"
    "4: DO NOT use introductory phrases (e.g., 'The image shows', 'The object is', 'The primary object is').\n\n"
    "Output only the final description of the product."
)
# VLM_REV_PROMPT_2 = (
#     "You are an assistant who identifies products in images for blind and low-vision users. Follow these guidelines and then output only the final description:\n"
#     "1. Identify the product (e.g., seasoning mix, soda, coffee).\n"
#     "2. Extract and label key attributes:\n"
#     "-- Product type (what it is).\n"
#     "-- Brand (e.g., Heinz, Coca-Cola, Starbucks).\n"
#     "-- Variety or variant (flavor, size, count, etc.).\n"
#     "-- Visual features (color, shape, distinctive markings).\n"
#     "3. Use precise, objective language. Avoid vague descriptors (“large,” “small,” “prominently,” “clearly”).\n"
#     "4. DO NOT mention camera artifacts (blur, glare) or cropping/partial visibility.\n"
#     "5. DO NOT prepend with “The image shows,” “The object is,” or similar phrases.\n\n"
#     "Output only the final description."
# )

print("Original Prompt")
print(VLM_ORIG_PROMPT)
print("-" * 100)
print("Revised Prompt 1")
print(VLM_REV_PROMPT)
print("-" * 100)
# print("Revised Prompt 2")
# print(VLM_REV_PROMPT_2)

## Add captions for all prompts

### Image Processing 

In [ ]:
def remove_transparency(im, bg_colour=(255, 255, 255)):
    """
    Remove transparency from an image.

    Args:
        im (PIL.Image.Image): Image to remove transparency from.
        bg_colour (tuple, optional): Background color to use for the transparent areas. Defaults to (255, 255, 255).

    Returns:
        PIL.Image.Image: Image with transparency removed.
    """
    # Only process if image has transparency (http://stackoverflow.com/a/1963146)
    if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
        # Need to convert to RGBA if LA format due to a bug in PIL (http://stackoverflow.com/a/1963146)
        alpha = im.convert("RGBA").split()[-1]

        # Create a new background image of our matt color.
        # Must be RGBA because paste requires both images have the same format
        # (http://stackoverflow.com/a/8720632  and  http://stackoverflow.com/a/9459208)
        bg = Image.new("RGBA", im.size, bg_colour + (255,))
        bg.paste(im, mask=alpha)
        return bg

    else:
        return im


def convert_to_base64(image_url):
    """
    Convert an image, specified by its url, to a PNG and return the base64 encoded string.

    Args:
        image_url (str): URL of the image to convert.

    Returns:
        str: Base64 encoded string of the image.
    """
    response = requests.get(image_url)
    image = Image.open(BytesIO(response.content))

    # remove transparency
    image = remove_transparency(image)

    with BytesIO() as f:
        image.save(f, format="PNG")
        f.seek(0)

        return base64.b64encode(f.read()).decode("utf-8")


def convert_to_png(image_url):
    response = requests.get(image_url)
    image = Image.open(BytesIO(response.content))

    # remove transparency
    image = remove_transparency(image)

    with BytesIO() as f:
        image.save(f, format="PNG")
        return f.getvalue()

### Captioning code

In [ ]:
def get_gpt_caption(
    image_url, openai_client, prompt, temperature=1.0, top_p=1.0, **kwargs
):
    # convert image_url to base54
    image_b64 = convert_to_base64(image_url)

    response = openai_client.responses.create(
        model=model_name,
        input=[
            {
                "role": "system",
                "content": [
                    {
                        "type": "input_text",
                        "text": prompt,
                    }
                ],
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_image",
                        "image_url": f"data:image/png;base64,{image_b64}",
                        "detail": "high",
                    }
                ],
            },
        ],
        text={"format": {"type": "text"}},
        reasoning={},
        tools=[],
        temperature=temperature,
        top_p=top_p,
        max_output_tokens=500,
        store=False,
    )

    if response.output_text is not None:
        return response.output_text
    else:
        return ""

In [ ]:
def get_gemini_caption(
    image_url, client, prompt, temperature=1.0, top_p=0.95, **kwargs
):
    # convert image to bytes
    image_b64 = convert_to_base64(image_url)

    # get caption
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            types.Content(
                role="user",
                parts=[
                    types.Part.from_bytes(
                        mime_type="image/png",
                        data=base64.b64decode(image_b64),
                    )
                ],
            ),
        ],
        config=types.GenerateContentConfig(
            temperature=temperature,
            top_p=top_p,
            max_output_tokens=500,
            thinking_config=types.ThinkingConfig(
                thinking_budget=0,
            ),
            media_resolution="MEDIA_RESOLUTION_MEDIUM",
            system_instruction=[types.Part.from_text(text=prompt)],
        ),
    )

    if response.text is not None:
        return response.text
    else:
        return ""

### General code

In [ ]:
models = [
    "gpt-4.1",
    "gemini-2.5-flash",
    "llama-90B-4bit",
    "molmo-72B-4bit",
]

model_settings = {
    # baseline
    "temp-1.0_topP-0.95": dict(temperature=1.0, top_p=0.95),
    # vary top_p
    "temp-1.0_topP-1.00": dict(temperature=1.0, top_p=1.00),
    "temp-1.0_topP-0.50": dict(temperature=1.0, top_p=0.50),
    "temp-1.0_topP-0.25": dict(temperature=1.0, top_p=0.25),
    "temp-1.0_topP-0.10": dict(temperature=1.0, top_p=0.10),
}

In [ ]:
for model_tag in models:
    # load relevant model
    if model_tag == "gpt-4.1":
        openai_client = OpenAI()
        openai_client.api_key = os.getenv("OPENAI_API_KEY")
        model_name = "gpt-4.1-2025-04-14"
    elif model_tag == "gemini-2.5-flash":
        # The client gets the API key from the environment variable `GEMINI_API_KEY`.
        google_client = genai.Client()
        model_name = "gemini-2.5-flash"
    elif model_tag == "llama-90B-4bit":
        model_name = "Llama-3.2-90B-Vision-Instruct-bnb-4bit"
        model_id = "unsloth/Llama-3.2-90B-Vision-Instruct-bnb-4bit"
        model = MllamaForConditionalGeneration.from_pretrained(
            model_id,
            torch_dtype=torch.bfloat16,
            device_map="auto",
        )
        processor = AutoProcessor.from_pretrained(model_id)

        # print model properties
        print("Model ID: ", model_id)
        print("Device: ", model.device)
        print("Dtype: ", model.dtype)
    elif model_tag == "molmo-72B-4bit":
        # For 2 x 24 GB. If using 1 x 48 GB or more (lucky you), you can just use device_map="auto"
        device_map = {
            "model.vision_backbone": 0,  # Seems to be required to not run out of memory at 48 GB
            "model.transformer.wte": 0,
            "model.transformer.ln_f": 0,
            "model.transformer.ff_out": 1,
        }
        # For 2 x 24 GB, this works for *only* 38 or 39. Any higher or lower and it'll either only work for 1 token of output or fail completely.
        switch_point = 38  # layer index to switch to second GPU
        device_map |= {
            f"model.transformer.blocks.{i}": 0 for i in range(0, switch_point)
        }
        device_map |= {
            f"model.transformer.blocks.{i}": 1 for i in range(switch_point, 80)
        }

        # model_name = "SeanScripts/Molmo-72B-0924-nf4"
        model_name = "kgarg0/Molmo-72B-0924-nf4-fixed"
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            use_safetensors=True,
            device_map=device_map,
            trust_remote_code=True,  # Required for Molmo at the moment.
        )
        model.model.vision_backbone.float()  # vision backbone needs to be in FP32 for this

        processor = AutoProcessor.from_pretrained(
            model_name,
            trust_remote_code=True,  # Required for Molmo at the moment.
        )

        # print model properties
        print("Model ID: ", model_name)
        print("Device: ", model.device)
        print("Dtype: ", model.dtype)

    print(f"Generating Captions for {model_tag} using {model_name}")

    for image_index, image_info in enumerate(tqdm(combined_sample_dict)):
        image_url = image_info["image_url"]
        image = Image.open(io.BytesIO(convert_to_png(image_info["image_url"])))

        for setting_name, values in model_settings.items():
            caption_name = f"{model_tag}_{setting_name}"

            try:
                # run the appropriate captioning code
                if model_tag == "gpt-4.1":
                    combined_sample_dict[image_index][caption_name] = get_gpt_caption(
                        image_url, openai_client, VLM_REV_PROMPT, **values
                    )
                elif model_tag == "gemini-2.5-flash":
                    combined_sample_dict[image_index][caption_name] = (
                        get_gemini_caption(
                            image_url, google_client, VLM_REV_PROMPT, **values
                        )
                    )
                elif model_tag == "llama-90B-4bit":
                    combined_sample_dict[image_index][caption_name] = get_llama_caption(
                        image,
                        model,
                        processor,
                        VLM_REV_PROMPT,
                        do_sample=True,
                        **values,
                    )
                elif model_tag == "molmo-72B-4bit":
                    combined_sample_dict[image_index][caption_name] = get_molmo_caption(
                        image,
                        model,
                        processor,
                        VLM_REV_PROMPT,
                        do_sample=True,
                        **values,
                    )
            except Exception as e:
                print(
                    f"Error processing image {image_index} ({image_info['image_url']}) for {model_tag} ({setting_name}): {e}"
                )
                continue

    # save intermediate file
    os.makedirs("./intermediate/", exist_ok=True)
    output_file = f"./intermediate/combined-sample-{LIMIT}-images_{model_tag}.json"
    print(f"Saving intermediate file for {model_tag} to {output_file}")
    with open(output_file, "w") as f:
        json.dump(combined_sample_dict, f)

    # clean-up
    if model_tag == "gpt-4.1":
        del openai_client
    elif model_tag == "gemini-2.5-flash":
        del google_client
    elif model_tag == "llama-90B-4bit":
        # clear cache and model objects
        del model_id, model, processor
        torch.cuda.empty_cache()
        gc.collect()
    elif model_tag == "molmo-72B-4bit":
        # clear cache and model objects
        del device_map, switch_point, model_name, model, processor
        torch.cuda.empty_cache()
        gc.collect()
    del model_name
    print("-" * 80)

In [ ]:
# write final json
os.makedirs("./final-output/", exist_ok=True)
with open(
    f"./final-output/combined-sample-{LIMIT}-images-captions_all-models_{datetime.now().strftime('%Y-%m-%d_%H:%M')}.json",
    "w",
) as f:
    json.dump(combined_sample_dict, f)

In [ ]:
# output csv
output_df = pd.DataFrame(combined_sample_dict)

# arrange columns
column_order = []
for model_tag in models:
    for setting_name, _ in model_settings.items():
        column_order.append(f"{model_tag}_{setting_name}")

# format output
output_df = output_df[
    [
        "id",
        "image_url",
        "image_preview",
        "type",
        "object",
        "product",
        "brand",
        "variety",
    ]
    + column_order
]
display(output_df.head())

# save
output_df.to_csv(
    f"./final-output/combined-sample-{LIMIT}-images-captions_all-models_{datetime.now().strftime('%Y-%m-%d_%H:%M')}.csv",
    index=False,
)

### GPT-4.1

In [ ]:
from openai import OpenAI

openai_client = OpenAI()
openai_client.api_key = os.getenv("OPENAI_API_KEY")
model_name = "gpt-4.1-2025-04-14"


def get_gpt_caption(
    image_url, openai_client, prompt, temperature=1.0, top_p=1.0, **kwargs
):
    # convert image_url to base54
    image_b64 = convert_to_base64(image_url)

    response = openai_client.responses.create(
        model=model_name,
        input=[
            {
                "role": "system",
                "content": [
                    {
                        "type": "input_text",
                        "text": prompt,
                    }
                ],
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_image",
                        "image_url": f"data:image/png;base64,{image_b64}",
                        "detail": "high",
                    }
                ],
            },
        ],
        text={"format": {"type": "text"}},
        reasoning={},
        tools=[],
        temperature=temperature,
        top_p=top_p,
        max_output_tokens=500,
        store=False,
    )

    if response.output_text is not None:
        return response.output_text
    else:
        return ""

In [ ]:
for image_index, image_info in enumerate(tqdm(combined_sample_dict)):
    # check if we already have captions
    if "gpt4.1_caption_temp_1" in image_info and "gpt4.1_caption_temp_0" in image_info:
        print(
            f"Skipping image {image_index} ({image_info['image_url']}) because it already has GPT-4.1 captions"
        )
        continue

    try:
        image_url = image_info["image_url"]

        # get caption from VLM_ORIG_PROMPT
        caption_rev = get_gpt_caption(
            image_url, openai_client, VLM_REV_PROMPT, temperature=1.0
        )
        caption_rev_2 = get_gpt_caption(
            image_url, openai_client, VLM_REV_PROMPT, temperature=0.0
        )

        # save captions to dataframe
        combined_sample_dict[image_index]["gpt4.1_caption_temp_1"] = caption_rev
        combined_sample_dict[image_index]["gpt4.1_caption_temp_0"] = caption_rev_2
    except Exception as e:
        print(f"Error processing image {image_index} ({image_info['image_url']}): {e}")
        continue
combined_sample_dict[0]

In [ ]:
# save
os.makedirs("./intermediate/", exist_ok=True)
with open(f"./intermediate/combined-sample-{LIMIT}-images_gpt4-1.json", "w") as f:
    json.dump(combined_sample_dict, f)

### Gemini

In [ ]:
from google import genai

# The client gets the API key from the environment variable `GEMINI_API_KEY`.
google_client = genai.Client()

In [ ]:
def get_gemini_caption(
    image_url, client, prompt, temperature=1.0, top_p=0.1, top_k=64, **kwargs
):
    # convert image to bytes
    image_b64 = convert_to_base64(image_url)

    # get caption
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            types.Content(
                role="user",
                parts=[
                    types.Part.from_bytes(
                        mime_type="image/png",
                        data=base64.b64decode(image_b64),
                    )
                ],
            ),
        ],
        config=types.GenerateContentConfig(
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            max_output_tokens=500,
            thinking_config=types.ThinkingConfig(
                thinking_budget=0,
            ),
            media_resolution="MEDIA_RESOLUTION_MEDIUM",
            system_instruction=[types.Part.from_text(text=prompt)],
        ),
    )

    if response.text is not None:
        return response.text
    else:
        return ""

In [ ]:
for image_index, image_info in enumerate(tqdm(combined_sample_dict)):
    # check if we already have captions
    if "gemini_caption_temp_1" in image_info and "gemini_caption_temp_0" in image_info:
        print(
            f"Skipping image {image_index} ({image_info['image_url']}) because it already has Gemini 2.5 Flash captions"
        )
        continue

    try:
        image_url = image_info["image_url"]

        # get caption from VLM_ORIG_PROMPT
        # caption_rev = get_gemini_caption(
        #     image_url, google_client, VLM_REV_PROMPT, temperature=1.0
        # )
        # caption_rev_2 = get_gemini_caption(
        #     image_url, google_client, VLM_REV_PROMPT, temperature=0.0
        # )

        caption_rev = get_gemini_caption(
            image_url, google_client, VLM_REV_PROMPT, temperature=1.0
        )
        caption_rev_2 = get_gemini_caption(
            image_url, google_client, VLM_REV_PROMPT, temperature=1.0, top_p=0.1
        )

        # save captions to dataframe
        combined_sample_dict[image_index]["gemini2.5flash_caption_temp_1"] = caption_rev
        combined_sample_dict[image_index]["gemini2.5flash_caption_temp_0"] = (
            caption_rev_2
        )
    except Exception as e:
        print(f"Error processing image {image_index} ({image_info['image_url']}): {e}")
        continue
combined_sample_dict[0]

In [ ]:
combined_sample_dict[0:2]

In [ ]:
# save
os.makedirs("./intermediate/", exist_ok=True)
with open(
    f"./intermediate/combined-sample-{LIMIT}-images_gemini-2.5-flash.json", "w"
) as f:
    json.dump(combined_sample_dict, f)

### Llama Big (90B, 4-bit Quantization)

In [ ]:
import torch
from transformers import MllamaForConditionalGeneration, AutoProcessor
from scripts.llama_captioner import generate_caption as get_llama_caption

model_name = "Llama-3.2-90B-Vision-Instruct-bnb-4bit"
model_id = "unsloth/Llama-3.2-90B-Vision-Instruct-bnb-4bit"
model = MllamaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(model_id)

# print model properties
print("Model ID: ", model_id)
print("Device: ", model.device)
print("Dtype: ", model.dtype)

In [ ]:
for image_index, image_info in enumerate(tqdm(combined_sample_dict)):
    # check if we already have captions
    if (
        "llama_big_caption_temp_1" in image_info
        and "llama_big_caption_rev_temp_0" in image_info
    ):
        print(
            f"Skipping image {image_index} ({image_info['image_url']}) because it already has Llama 3.2 90B captions"
        )
        continue

    try:
        # load image with pillow
        image = Image.open(io.BytesIO(convert_to_png(image_info["image_url"])))

        # get caption from VLM_ORIG_PROMPT
        caption_rev = get_llama_caption(
            image, model, processor, VLM_REV_PROMPT, temperature=1.0, do_sample=True
        )
        caption_rev_2 = get_llama_caption(
            image, model, processor, VLM_REV_PROMPT, temperature=0.0, do_sample=False
        )

        # save captions to dict
        combined_sample_dict[image_index]["llama3.290b_caption_temp_1"] = caption_rev
        combined_sample_dict[image_index]["llama3.290b_caption_temp_0"] = caption_rev_2
    except Exception as e:
        print(f"Error processing image {image_index} ({image_info['image_url']}): {e}")
        continue
combined_sample_dict[0]

In [ ]:
# save
os.makedirs("./intermediate/", exist_ok=True)
with open(
    f"./intermediate/combined-sample-{LIMIT}-images_llama-3.2-90b.json", "w"
) as f:
    json.dump(combined_sample_dict, f)

In [ ]:
# clear cache and model objects
del model, processor
torch.cuda.empty_cache()
gc.collect()

### Molmo Big (72B parameter model, 4-bit quantization)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor
from scripts.molmo_captioner import generate_caption as get_molmo_caption
from PIL import Image

# For 2 x 24 GB. If using 1 x 48 GB or more (lucky you), you can just use device_map="auto"
device_map = {
    "model.vision_backbone": 0,  # Seems to be required to not run out of memory at 48 GB
    "model.transformer.wte": 0,
    "model.transformer.ln_f": 0,
    "model.transformer.ff_out": 1,
}
# For 2 x 24 GB, this works for *only* 38 or 39. Any higher or lower and it'll either only work for 1 token of output or fail completely.
switch_point = 38  # layer index to switch to second GPU
device_map |= {f"model.transformer.blocks.{i}": 0 for i in range(0, switch_point)}
device_map |= {f"model.transformer.blocks.{i}": 1 for i in range(switch_point, 80)}

# model_name = "SeanScripts/Molmo-72B-0924-nf4"
model_name = "kgarg0/Molmo-72B-0924-nf4-fixed"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    use_safetensors=True,
    device_map=device_map,
    trust_remote_code=True,  # Required for Molmo at the moment.
)
model.model.vision_backbone.float()  # vision backbone needs to be in FP32 for this

processor = AutoProcessor.from_pretrained(
    model_name,
    trust_remote_code=True,  # Required for Molmo at the moment.
)

# print model properties
print("Model ID: ", model_name)
print("Device: ", model.device)
print("Dtype: ", model.dtype)

In [ ]:
for image_index, image_info in enumerate(tqdm(combined_sample_dict)):
    # check if we already have captions
    if (
        "molmo_big_caption_temp_1" in image_info
        and "molmo_big_caption_temp_0" in image_info
    ):
        print(
            f"Skipping image {image_index} ({image_info['image_url']}) because it already has Molmo 72B captions"
        )
        continue

    try:
        # load image with pillow
        image = Image.open(io.BytesIO(convert_to_png(image_info["image_url"])))

        # get caption from VLM_ORIG_PROMPT
        caption_rev = get_molmo_caption(
            image, model, processor, VLM_REV_PROMPT, temperature=1.0, do_sample=True
        )
        caption_rev_2 = get_molmo_caption(
            image, model, processor, VLM_REV_PROMPT, temperature=0.0, do_sample=False
        )

        # save captions to dict
        combined_sample_dict[image_index]["molmo72b_caption_temp_1"] = caption_rev
        combined_sample_dict[image_index]["molmo72b_caption_temp_0"] = caption_rev_2
    except Exception as e:
        print(f"Error processing image {image_index} ({image_info['image_url']}): {e}")
        continue
combined_sample_dict[0]

In [ ]:
# clear cache and model objects
del model, processor
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# save
os.makedirs("./intermediate/", exist_ok=True)
with open(f"./intermediate/combined-sample-{LIMIT}-images_molmo-72b.json", "w") as f:
    json.dump(combined_sample_dict, f)

## Output final json and CSV

In [ ]:
# write final json
os.makedirs("./final-output/", exist_ok=True)
with open(
    f"./final-output/combined-sample-{LIMIT}-images-captions_all-models_{datetime.now().strftime('%Y-%m-%d_%H:%M')}.json",
    "w",
) as f:
    json.dump(combined_sample_dict, f)

In [ ]:
# output csv
output_df = pd.DataFrame(combined_sample_dict)

# arrange columns
column_order = []
for model in [
    # "gpt",
    "gpt4.1",
    "gemini2.5flash",
    # "llama",
    "llama3.290b",
    # "molmo",
    "molmo72b",
]:
    for prompt in ["temp_1", "temp_0"]:
        column_order.append(f"{model}_caption_{prompt}")

output_df = output_df[
    [
        "id",
        "image_url",
        "image_preview",
        "type",
        "object",
        "product",
        "brand",
        "variety",
    ]
    + column_order
]
output_df.head()

# save
output_df.to_csv(
    f"./final-output/combined-sample-{LIMIT}-images-captions_all-models_{datetime.now().strftime('%Y-%m-%d_%H:%M')}.csv",
    index=False,
)

# Old Code

## GPT-4o

In [ ]:
# from openai import OpenAI

# openai_client = OpenAI()
# openai_client.api_key = os.getenv("OPENAI_API_KEY")
# model_name = "gpt-4o-2024-08-06"


# def get_gpt_caption(image_url, openai_client, prompt, temperature=1.0):
#     response = openai_client.responses.create(
#         model=model_name,
#         input=[
#             {
#                 "role": "system",
#                 "content": [
#                     {
#                         "type": "input_text",
#                         "text": prompt,
#                     }
#                 ],
#             },
#             {
#                 "role": "user",
#                 "content": [
#                     {
#                         "type": "input_image",
#                         "image_url": f"data:image/png;base64,{image_url}",
#                         "detail": "high",
#                     }
#                 ],
#             },
#         ],
#         text={"format": {"type": "text"}},
#         reasoning={},
#         tools=[],
#         temperature=temperature,
#         max_output_tokens=300,
#         top_p=1,
#         store=False,
#     )

#     if response.output_text is not None:
#         return response.output_text
#     else:
#         return ""

In [ ]:
# for image_index, image_info in enumerate(tqdm(matched_product_df_dict)):
#     # check if we already have captions
#     if (
#         "gpt_caption_orig" in image_info
#         and "gpt_caption_rev" in image_info
#         and "gpt_caption_rev_2" in image_info
#     ):
#         print(
#             f"Skipping image {image_index} ({image_info['image_url']}) because it already has GPT-4o captions"
#         )
#         continue

#     try:
#         image_url = image_info["image_url"]
#         image_b64 = convert_to_base64(image_url)

#         # get caption from VLM_ORIG_PROMPT
#         caption_orig = get_gpt_caption(
#             image_b64, openai_client, VLM_ORIG_PROMPT, temperature=1.0
#         )
#         caption_rev = get_gpt_caption(
#             image_b64, openai_client, VLM_REV_PROMPT, temperature=1.0
#         )
#         caption_rev_2 = get_gpt_caption(
#             image_b64, openai_client, VLM_REV_PROMPT_2, temperature=1.0
#         )

#         # save captions to dataframe
#         matched_product_df_dict[image_index]["gpt_caption_orig"] = caption_orig
#         matched_product_df_dict[image_index]["gpt_caption_rev"] = caption_rev
#         matched_product_df_dict[image_index]["gpt_caption_rev_2"] = caption_rev_2
#     except Exception as e:
#         print(f"Error processing image {image_index} ({image_info['image_url']}): {e}")
#         continue
# matched_product_df_dict[0]

In [ ]:
# # save
# os.makedirs("./intermediate/", exist_ok=True)
# with open("./intermediate/matched-product-captions-gpt4o.json", "w") as f:
#     json.dump(matched_product_df_dict, f)

## Llama

In [ ]:
# import torch
# from transformers import MllamaForConditionalGeneration, AutoProcessor
# from scripts.llama_captioner import generate_caption as get_llama_caption

# model_name = "Llama-3.2-11B-Vision-Instruct"
# model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"
# model = MllamaForConditionalGeneration.from_pretrained(
#     model_id,
#     torch_dtype=torch.bfloat16,
#     device_map="auto",
# )
# processor = AutoProcessor.from_pretrained(model_id)

# # print model properties
# print("Model ID: ", model_id)
# print("Device: ", model.device)
# print("Dtype: ", model.dtype)

In [ ]:
# for image_index, image_info in enumerate(tqdm(matched_product_df_dict)):
#     # check if we already have captions
#     if (
#         "llama_caption_orig" in image_info
#         and "llama_caption_rev" in image_info
#         and "llama_caption_rev_2" in image_info
#     ):
#         print(
#             f"Skipping image {image_index} ({image_info['image_url']}) because it already has Llama captions"
#         )
#         continue

#     try:
#         # load image with pillow
#         image = Image.open(io.BytesIO(convert_to_png(image_info["image_url"])))

#         # get caption from VLM_ORIG_PROMPT
#         caption_orig = get_llama_caption(
#             image, model, processor, VLM_ORIG_PROMPT, temperature=1.0
#         )
#         caption_rev = get_llama_caption(
#             image, model, processor, VLM_REV_PROMPT, temperature=1.0
#         )
#         caption_rev_2 = get_llama_caption(
#             image, model, processor, VLM_REV_PROMPT_2, temperature=1.0
#         )

#         # save captions to dict
#         matched_product_df_dict[image_index]["llama_caption_orig"] = caption_orig
#         matched_product_df_dict[image_index]["llama_caption_rev"] = caption_rev
#         matched_product_df_dict[image_index]["llama_caption_rev_2"] = caption_rev_2
#     except Exception as e:
#         print(f"Error processing image {image_index} ({image_info['image_url']}): {e}")
#         continue
# matched_product_df_dict[0]

In [ ]:
# # save
# os.makedirs("./intermediate/", exist_ok=True)
# with open("./intermediate/matched-product-captions-llama.json", "w") as f:
#     json.dump(matched_product_df_dict, f)

In [ ]:
# # clear cache and model objects
# torch.cuda.empty_cache()
# del model, processor
# gc.collect()

## Molmo

In [ ]:
# import torch
# from transformers import AutoModelForCausalLM, AutoProcessor
# from scripts.molmo_captioner import generate_caption as get_molmo_caption

# model_name = "Molmo-7B-D-0924"
# model_id = "allenai/Molmo-7B-D-0924"

# # load model
# processor = AutoProcessor.from_pretrained(
#     model_id,
#     trust_remote_code=True,
#     torch_dtype="auto",
#     device_map="auto",
# )

# model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     trust_remote_code=True,
#     torch_dtype="auto",
#     device_map="auto",
# )

# # print model properties
# print("Model ID: ", model_id)
# print("Device: ", model.device)
# print("Dtype: ", model.dtype)

In [ ]:
# for image_index, image_info in enumerate(tqdm(matched_product_df_dict)):
#     # check if we already have captions
#     if (
#         "molmo_caption_orig" in image_info
#         and "molmo_caption_rev" in image_info
#         and "molmo_caption_rev_2" in image_info
#     ):
#         print(
#             f"Skipping image {image_index} ({image_info['image_url']}) because it already has Molmo captions"
#         )
#         continue

#     try:
#         # load image with pillow
#         image = Image.open(io.BytesIO(convert_to_png(image_info["image_url"])))

#         # get caption from VLM_ORIG_PROMPT
#         caption_orig = get_molmo_caption(
#             image, model, processor, VLM_ORIG_PROMPT, temperature=1.0
#         )
#         caption_rev = get_molmo_caption(
#             image, model, processor, VLM_REV_PROMPT, temperature=1.0
#         )
#         caption_rev_2 = get_molmo_caption(
#             image, model, processor, VLM_REV_PROMPT_2, temperature=1.0
#         )

#         # save captions to dict
#         matched_product_df_dict[image_index]["molmo_caption_orig"] = caption_orig
#         matched_product_df_dict[image_index]["molmo_caption_rev"] = caption_rev
#         matched_product_df_dict[image_index]["molmo_caption_rev_2"] = caption_rev_2
#     except Exception as e:
#         print(f"Error processing image {image_index} ({image_info['image_url']}): {e}")
#         continue
# matched_product_df_dict[0]

In [ ]:
# # save
# os.makedirs("./intermediate/", exist_ok=True)
# with open("./intermediate/matched-product-captions-molmo.json", "w") as f:
#     json.dump(matched_product_df_dict, f)

In [ ]:
# # clear cache and model objects
# torch.cuda.empty_cache()
# del model, processor
# gc.collect()